## 1. Kiểm tra môi trường

In [1]:
import sys
import io
import json
import inspect
import importlib.util
import subprocess
from pathlib import Path

for package in ("numpy", "torch"):
    if importlib.util.find_spec(package) is None:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import numpy as np
import torch

try:
    from google.colab import files
    IN_COLAB = True
except ImportError:
    IN_COLAB = False

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
assert (torch.ones(2, device=device) + 1).tolist() == [2.0, 2.0]

## 2. Tham số và thư mục

In [2]:
PROJECT_DIR = (Path("/content") if IN_COLAB else Path.cwd()) / "Gaussian2D"
DATA_DIR = PROJECT_DIR / "data"
SRC_DIR = PROJECT_DIR / "src"
for folder in (DATA_DIR, SRC_DIR):
    folder.mkdir(parents=True, exist_ok=True)

MU = np.array([0.0, 0.0], dtype=np.float64)
C = np.array([[0.3, -0.5], [-0.5, 2.0]], dtype=np.float64)
LOW = np.array([-2.0, -4.0])
HIGH = np.array([2.0, 4.0])
ROOT_SEED = 20261024
SPLIT_SIZES = {"train_full": 2000, "val": 400, "test": 100}
SPLIT_IDS = {"train_full": 0, "val": 1, "test": 2}
print("Thư mục:", PROJECT_DIR)

Thư mục: /content/Gaussian2D


## 3. Hàm Gaussian bằng NumPy

Nhận đầu vào có chiều cuối bằng 2. Công thức dùng đúng ma trận C đã khai báo.

In [3]:
def gaussian_pdf_numpy(x):
    x = np.asarray(x, dtype=np.float64)
    if x.ndim == 0 or x.shape[-1] != 2 or not np.isfinite(x).all():
        raise ValueError("x phải có dạng (..., 2) và không chứa NaN/Inf.")
    delta = x - MU
    quadratic = np.einsum("...i,ij,...j->...", delta, np.linalg.inv(C), delta)
    return np.exp(-0.5 * quadratic) / (2 * np.pi * np.sqrt(np.linalg.det(C)))

print("det(C) =", np.linalg.det(C))
print("f(0,0) =", gaussian_pdf_numpy([0, 0]))

det(C) = 0.3499999999999999
f(0,0) = 0.26902095463038056


## 4. Hàm Gaussian bằng PyTorch

In [4]:
def gaussian_pdf_torch(x):
    if not isinstance(x, torch.Tensor):
        x = torch.as_tensor(x, dtype=torch.float64)
    if x.dtype not in (torch.float32, torch.float64):
        x = x.to(torch.float64)
    if x.ndim == 0 or x.shape[-1] != 2 or not torch.isfinite(x).all():
        raise ValueError("x phải có dạng (..., 2) và không chứa NaN/Inf.")
    covariance = x.new_tensor(C.tolist())
    delta = x - x.new_tensor(MU.tolist())
    quadratic = torch.einsum("...i,ij,...j->...", delta, torch.linalg.inv(covariance), delta)
    return torch.exp(-0.5 * quadratic) / (2 * torch.pi * torch.sqrt(torch.linalg.det(covariance)))

print("PyTorch f(0,0) =", gaussian_pdf_torch([0, 0]).item())

PyTorch f(0,0) = 0.26902095463038056


## 5. Hàm sinh dữ liệu

X có shape `(N,2)`, y có shape `(N,1)`. Các điểm X được lấy đều trên miền, vì vậy hiệp phương sai thực nghiệm của X không cần bằng C.

In [5]:
def generate_dataset(n_samples, split):
    if isinstance(n_samples, bool) or not isinstance(n_samples, (int, np.integer)) or n_samples <= 0:
        raise ValueError("Số mẫu phải là số nguyên dương.")
    if split not in SPLIT_IDS:
        raise ValueError("Split phải là train_full, val hoặc test.")
    seed = np.random.SeedSequence(ROOT_SEED, spawn_key=(SPLIT_IDS[split],))
    rng = np.random.Generator(np.random.PCG64(seed))
    x = rng.uniform(LOW, HIGH, size=(n_samples, 2))
    y = gaussian_pdf_numpy(x).reshape(-1, 1)
    return x, y

## 6. Hàm ghi và đọc CSV

In [6]:
def save_csv(path, x, y):
    buffer = io.StringIO()
    np.savetxt(buffer, np.column_stack((x, y)), delimiter=",", fmt="%.17g",
               header="x1,x2,y", comments="", newline="\n")
    content = buffer.getvalue().encode("utf-8")
    path = Path(path)
    if path.exists():
        if path.read_bytes() != content:
            raise FileExistsError(f"{path.name} có nội dung khác. Hãy dùng thư mục mới.")
    else:
        path.parent.mkdir(parents=True, exist_ok=True)
        path.write_bytes(content)


def load_csv(path):
    with Path(path).open(encoding="utf-8-sig") as handle:
        if handle.readline().strip() != "x1,x2,y":
            raise ValueError("CSV phải có header x1,x2,y.")
        table = np.loadtxt(handle, delimiter=",", ndmin=2)
    if table.shape[1] != 3 or len(table) == 0 or not np.isfinite(table).all():
        raise ValueError("CSV thiếu dữ liệu, sai số cột hoặc chứa NaN/Inf.")
    return table[:, :2].copy(), table[:, 2:3].copy()


def load_torch_csv(path):
    x, y = load_csv(path)
    return torch.tensor(x, dtype=torch.float32), torch.tensor(y, dtype=torch.float32)

## 7. Tạo ba tập và xuất CSV


In [7]:
datasets = {}
for split, count in SPLIT_SIZES.items():
    x, y = generate_dataset(count, split)
    save_csv(DATA_DIR / f"{split}.csv", x, y)
    datasets[split] = load_csv(DATA_DIR / f"{split}.csv")
    print(f"{split}.csv: {count} mẫu | X {x.shape} | y {y.shape}")

train_full.csv: 2000 mẫu | X (2000, 2) | y (2000, 1)
val.csv: 400 mẫu | X (400, 2) | y (400, 1)
test.csv: 100 mẫu | X (100, 2) | y (100, 1)


## 8. Kiểm tra nghiệm thu

Đối chiếu nhãn với Gaussian có sẵn của PyTorch và kiểm tra dữ liệu đọc từ CSV.

In [8]:
np.testing.assert_array_equal(C, [[0.3, -0.5], [-0.5, 2.0]])
np.testing.assert_array_equal(MU, [0.0, 0.0])
assert (np.linalg.eigvalsh(C) > 0).all()
reference = torch.distributions.MultivariateNormal(
    torch.zeros(2, dtype=torch.float64),
    covariance_matrix=torch.tensor([[0.3, -0.5], [-0.5, 2.0]], dtype=torch.float64),
)
report = {}
for split, (x, y) in datasets.items():
    assert x.shape == (SPLIT_SIZES[split], 2) and y.shape == (SPLIT_SIZES[split], 1)
    assert np.isfinite(x).all() and np.isfinite(y).all()
    assert ((x >= LOW) & (x <= HIGH)).all()
    assert len(np.unique(x, axis=0)) == len(x)
    xt = torch.tensor(x, dtype=torch.float64)
    expected = reference.log_prob(xt).exp().numpy()
    np.testing.assert_allclose(y[:, 0], expected, rtol=1e-12, atol=1e-15)
    np.testing.assert_allclose(gaussian_pdf_torch(xt).numpy(), expected, rtol=1e-12, atol=1e-15)
    repeated_x, repeated_y = generate_dataset(len(x), split)
    np.testing.assert_array_equal(x, repeated_x)
    np.testing.assert_array_equal(y, repeated_y)
    report[split] = {"samples": len(x), "status": "PASS"}
    print(f"PASS: {split} — số mẫu, miền, NaN/Inf, công thức và seed.")

PASS: train_full — số mẫu, miền, NaN/Inf, công thức và seed.
PASS: val — số mẫu, miền, NaN/Inf, công thức và seed.
PASS: test — số mẫu, miền, NaN/Inf, công thức và seed.


### Kiểm tra tách biệt giữa các tập

In [9]:
from itertools import combinations

for left, right in combinations(SPLIT_SIZES, 2):
    common = set(map(tuple, datasets[left][0])) & set(map(tuple, datasets[right][0]))
    assert not common
    print(f"{left} / {right}: {len(common)} điểm trùng")

test_before = generate_dataset(100, "test")
generate_dataset(17, "train_full")
test_after = generate_dataset(100, "test")
assert all(np.array_equal(a, b) for a, b in zip(test_before, test_after))

_ = (PROJECT_DIR / "validation.json").write_text(
    json.dumps(report, ensure_ascii=False, indent=2), encoding="utf-8"
)

train_full / val: 0 điểm trùng
train_full / test: 0 điểm trùng
val / test: 0 điểm trùng


## 9. Xuất mã nguồn thành `src/data.py`



In [10]:
source = "from pathlib import Path\nimport io\nimport numpy as np\nimport torch\n\n"
for name in ("MU", "C", "LOW", "HIGH"):
    source += f"{name} = np.array({globals()[name].tolist()!r}, dtype=np.float64)\n"
for name in ("ROOT_SEED", "SPLIT_SIZES", "SPLIT_IDS"):
    source += f"{name} = {globals()[name]!r}\n"
functions = [gaussian_pdf_numpy, gaussian_pdf_torch, generate_dataset,
             save_csv, load_csv, load_torch_csv]
source += "\n\n" + "\n\n".join(inspect.getsource(function) for function in functions)
(SRC_DIR / "__init__.py").touch()
(SRC_DIR / "data.py").write_text(source, encoding="utf-8")
print("Đã lưu:", SRC_DIR / "data.py")

Đã lưu: /content/Gaussian2D/src/data.py


## 10. Đọc CSV để viết tiếp MLP

Train và validation được nạp thành tensor float32.

In [11]:
from torch.utils.data import DataLoader, TensorDataset

X_train, y_train = load_torch_csv(DATA_DIR / "train_full.csv")
X_val, y_val = load_torch_csv(DATA_DIR / "val.csv")
train_loader = DataLoader(
    TensorDataset(X_train, y_train), batch_size=64, shuffle=True,
    generator=torch.Generator().manual_seed(ROOT_SEED),
)
val_loader = DataLoader(TensorDataset(X_val, y_val), batch_size=64, shuffle=False)
print("Train:", tuple(X_train.shape), tuple(y_train.shape))
print("Val:  ", tuple(X_val.shape), tuple(y_val.shape))

Train: (2000, 2) (2000, 1)
Val:   (400, 2) (400, 1)
